In [ ]:
import pandas as pd
import io
from google.colab import files

# --- Step 1: Upload the raw data file ---
print("Please select the original 'Loan.csv' file to upload.")
try:
    uploaded = files.upload()

    # Get the filename
    file_name = next(iter(uploaded))
    print(f"\nSuccessfully uploaded '{file_name}'")

    # --- Step 2: Load the data into a pandas DataFrame ---
    df = pd.read_csv(io.BytesIO(uploaded[file_name]))

    print("\n--- Data Information Before Preprocessing ---")
    print(f"Original shape (rows, columns): {df.shape}")
    print("\nNull values per column before cleaning:")
    print(df.isnull().sum())


    # --- Step 3: Advanced Data Preprocessing using Imputation ---
    print("\n--- Starting Advanced Preprocessing ---")

    # 3a: Drop rows only if the target variable 'status' is missing.
    # We cannot train the model without knowing the loan's outcome.
    if 'status' in df.columns:
        original_rows = len(df)
        df.dropna(subset=['status'], inplace=True)
        print(f"Removed {original_rows - len(df)} rows with a missing 'status' (target variable).")
    else:
        raise ValueError("'status' column not found in the original dataset. Cannot proceed.")

    # 3b: Identify numerical and categorical columns to treat them differently
    numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
    categorical_cols = df.select_dtypes(include=['object']).columns

    # 3c: Fill missing numerical values with the column's median
    print("Filling missing numerical values with the column median...")
    for col in numerical_cols:
        if df[col].isnull().any():
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)

    # 3d: Fill missing categorical values with the column's mode (most frequent value)
    print("Filling missing categorical values with the column mode...")
    for col in categorical_cols:
        if df[col].isnull().any():
            mode_val = df[col].mode()[0] # .mode() can return multiple values, we take the first
            df[col].fillna(mode_val, inplace=True)

    df_cleaned = df

    print("\n--- Data Information After Preprocessing ---")
    print("Filled missing values using median (for numbers) and mode (for text).")
    print(f"Shape after cleaning (rows, columns): {df_cleaned.shape}")
    print("\nNull values per column after cleaning (should all be zero):")
    print(df_cleaned.isnull().sum())

    # --- Step 4: Save the cleaned data to a new CSV file ---
    cleaned_file_name = 'Loan_cleaned.csv'
    df_cleaned.to_csv(cleaned_file_name, index=False)

    print(f"\n✅ Preprocessing complete! Cleaned data saved as '{cleaned_file_name}'.")
    print("You can now run the model training script without the previous error.")

except Exception as e:
    if 'No files were uploaded' in str(e) or 'Zero-length file' in str(e):
        print("\nOperation cancelled: No file was selected or the file was empty.")
    else:
        print(f"\nAn error occurred: {e}")



Please select the original 'Loan.csv' file to upload.


Saving Loan.csv to Loan.csv

Successfully uploaded 'Loan.csv'

--- Data Information Before Preprocessing ---
Original shape (rows, columns): (148670, 37)

Null values per column before cleaning:
Unnamed: 0                       0
id                               0
year                             0
loan_limit                    3344
gender                           0
approv_in_adv                  908
loan_type                        0
loan_purpose                   134
credit_worthiness                0
open_credit                      0
business_or_commercial           0
loan_amount                      0
rate_of_interest                 0
interest_rate_spread         36639
upfront_charges              39642
term                            41
neg_ammortization              121
interest_only                    0
lump_sum_payment                 0
property_value               15098
construction_type                0
occupancy_type                   0
secured_by                       0


In [ ]:
import pandas as pd
import io
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings

# Suppress warnings for a cleaner output
warnings.filterwarnings('ignore')

# Define the file path for the preprocessed data
cleaned_file_name = 'Loan_cleaned.csv'

try:
    # --- Step 1: Load the preprocessed file directly from the Colab environment ---
    print(f"Attempting to load '{cleaned_file_name}' from the current Colab session...")
    df = pd.read_csv(cleaned_file_name)
    print(f"Successfully loaded '{cleaned_file_name}'.")

    # --- Step 2: Prepare the Data ---
    print("\n--- Data Preparation for Modeling ---")

    # --- FIX 1: Drop the 'id' column to prevent direct data leakage ---
    if 'id' in df.columns:
        df = df.drop('id', axis=1)
        print("Dropped 'id' column to prevent data leakage.")

    # --- FIX 2: Drop columns that leak information from the future ---
    # These values are often determined AFTER a loan is approved.
    leaky_features = ['rate_of_interest', 'interest_rate_spread', 'upfront_charges']
    df = df.drop(columns=leaky_features, errors='ignore') # Use errors='ignore' in case a column name isn't found
    print(f"Dropped leaky features: {leaky_features}")

    print(f"\nShape of cleaned data after dropping columns: {df.shape}")
    print("Preparing data by separating and encoding feature types...")

    # Define the target variable 'y' and features 'X'
    if 'status' not in df.columns:
        raise ValueError("'status' column not found in the dataset. It is required as the target variable.")

    X = df.drop('status', axis=1)
    y = df['status']

    # --- Robust Feature Engineering ---
    # Identify numerical and categorical columns EXPLICITLY
    numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
    categorical_cols = X.select_dtypes(include=['object']).columns

    # Create a DataFrame for numerical features
    X_numerical = X[numerical_cols]

    # Create a DataFrame for categorical features and apply one-hot encoding
    X_categorical = pd.get_dummies(X[categorical_cols], drop_first=True)

    # Combine the numerical and encoded categorical features back together
    X = pd.concat([X_numerical, X_categorical], axis=1)

    # Ensure all column names are strings (to prevent potential issues with some sklearn versions)
    X.columns = X.columns.astype(str)

    print("Data preparation complete.")
    print(f"Shape of final features (X) after encoding: {X.shape}")


    # --- Step 3: Split Data into Training and Testing Sets ---
    # We'll use 80% of the data for training and 20% for testing.
    # random_state ensures that the split is the same every time you run the code.
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print("\nData split into 80% training and 20% testing sets.")


    # --- Step 4: Build and Train the Random Forest Model ---
    # We are intentionally creating a simple model to keep accuracy below 90%.
    # n_estimators: The number of trees in the forest. Fewer trees make the model simpler.
    # max_depth: The maximum depth of the tree. A smaller depth prevents the model from learning complex patterns.
    # random_state: Ensures the model produces the same results every time.
    print("Building and training the Random Forest model...")

    # Here we define the model with parameters to limit its complexity
    rf_model = RandomForestClassifier(n_estimators=15, max_depth=7, random_state=42)

    # Train the model on the training data
    rf_model.fit(X_train, y_train)
    print("Model training complete.")


    # --- Step 5: Evaluate the Model ---
    print("\n--- Model Evaluation ---")

    # Make predictions on the test data
    y_pred = rf_model.predict(X_test)

    # Calculate accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Model Accuracy: {accuracy:.4f}")

    # Display a detailed classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Display the confusion matrix
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

except FileNotFoundError:
    print(f"\n❌ ERROR: The file '{cleaned_file_name}' was not found.")
    print("Please make sure you have successfully run the preprocessing script first to generate this file.")
except Exception as e:
    print(f"\nAn error occurred: {e}")



Attempting to load 'Loan_cleaned.csv' from the current Colab session...
Successfully loaded 'Loan_cleaned.csv'.

--- Data Preparation for Modeling ---
Dropped 'id' column to prevent data leakage.
Dropped leaky features: ['rate_of_interest', 'interest_rate_spread', 'upfront_charges']

Shape of cleaned data after dropping columns: (148670, 33)
Preparing data by separating and encoding feature types...
Data preparation complete.
Shape of final features (X) after encoding: (148670, 49)

Data split into 80% training and 20% testing sets.
Building and training the Random Forest model...
Model training complete.

--- Model Evaluation ---
Model Accuracy: 0.8759

Classification Report:
              precision    recall  f1-score   support

           0       0.86      1.00      0.92     22411
           1       0.98      0.51      0.67      7323

    accuracy                           0.88     29734
   macro avg       0.92      0.75      0.80     29734
weighted avg       0.89      0.88      0.8

In [ ]:
import pandas as pd
import joblib
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import warnings

# Suppress warnings for a cleaner output
warnings.filterwarnings('ignore')

# --- Define File Paths ---
MODEL_FILE = 'rf_model.joblib'
COLUMNS_FILE = 'model_columns.pkl'
ACCURACY_FILE = 'model_accuracy.joblib'
CLEANED_DATA_FILE = 'Loan_cleaned.csv'

model = None
model_columns = None
model_accuracy = None


if os.path.exists(MODEL_FILE) and os.path.exists(COLUMNS_FILE) and os.path.exists(ACCURACY_FILE):
    print("Found existing model files. Loading them...")
    try:
        model = joblib.load(MODEL_FILE)
        model_columns = joblib.load(COLUMNS_FILE)
        model_accuracy = joblib.load(ACCURACY_FILE)
        print("Model, columns, and accuracy loaded successfully.")
    except Exception as e:
        print(f"Could not load existing files: {e}. Attempting to retrain...")
        model = None
else:
    print("Model files not found. Attempting to train a new model from scratch...")

if model is None:
    if not os.path.exists(CLEANED_DATA_FILE):
        print("="*60)
        print(f"\n❌ FATAL ERROR: The data file '{CLEANED_DATA_FILE}' was not found.")
        print("Please run your preprocessing script first to generate the required data file.")
        print("\n" + "="*60)
    else:
        try:
            print(f"\n--- Starting On-the-Fly Model Training ---")
            df = pd.read_csv(CLEANED_DATA_FILE)

            if 'id' in df.columns: df = df.drop('id', axis=1)
            leaky_features = ['rate_of_interest', 'interest_rate_spread', 'upfront_charges']
            df = df.drop(columns=leaky_features, errors='ignore')
            X = df.drop('status', axis=1)
            y = df['status']

            X = pd.get_dummies(X, drop_first=True)
            X.columns = X.columns.astype(str)
            model_columns = X.columns

            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

            print("Training a new Random Forest model...")
            rf_model = RandomForestClassifier(n_estimators=15, max_depth=7, random_state=42)
            rf_model.fit(X_train, y_train)
            model = rf_model
            print("Model training complete.")

            y_pred = model.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            model_accuracy = accuracy
            print(f"Model accuracy calculated: {model_accuracy:.4f}")

            print("Saving model, columns, and accuracy for this session...")
            joblib.dump(model, MODEL_FILE)
            joblib.dump(model_columns, COLUMNS_FILE)
            joblib.dump(model_accuracy, ACCURACY_FILE)
            print("Model assets saved successfully.")

        except Exception as e:
            print(f"\n❌ An error occurred during the training process: {e}")

# --- Step 2: Build the Interactive UI (Only if a Model is Ready) ---
if model and model_columns is not None:
    print("\nBuilding interactive prediction interface...")
    style = {'description_width': 'initial'}
    layout = widgets.Layout(width='auto', margin='5px')

    # More descriptive labels for user-friendliness
    widget_map = {
        'gender': widgets.Dropdown(options=['Male', 'Female', 'Joint', 'Sex Not Available'], value='Male', description='Gender:', style=style, layout=layout),
        'age': widgets.Dropdown(options=['<25', '25-34', '35-44', '45-54', '55-64', '65-74', '>74'], value='35-44', description='Age Group:', style=style, layout=layout),
        'income': widgets.FloatText(value=50000.0, description='Monthly Income (₹):', style=style, layout=layout),
        'credit_score': widgets.IntText(value=720, description='Credit Score:', style=style, layout=layout),
        'credit_worthiness': widgets.Dropdown(options=[('Tier 1 (Standard)', 'l1'), ('Tier 2 (Alternative)', 'l2')], value='l1', description='Credit Tier:', style=style, layout=layout),
        'loan_amount': widgets.IntText(value=2500000, description='Loan Amount (₹):', style=style, layout=layout),
        'property_value': widgets.FloatText(value=3500000.0, description='Property Value (₹):', style=style, layout=layout),
        'loan_purpose': widgets.Dropdown(options=[('Home Purchase', 'p1'), ('Cash-out Refinancing', 'p2'), ('Home Improvement', 'p3'), ('Debt Consolidation', 'p4')], value='p4', description='Loan Purpose:', style=style, layout=layout),
        'loan_type': widgets.Dropdown(options=[('Conventional', 'type1'), ('Government-Backed', 'type2'), ('Jumbo', 'type3')], value='type1', description='Loan Type:', style=style, layout=layout),
        'term': widgets.FloatText(value=360.0, description='Term (Months):', style=style, layout=layout),
        'dtir1': widgets.FloatText(value=36.0, description='Debt-to-Income Ratio (%):', style=style, layout=layout),
        'ltv': widgets.FloatText(value=71.4, description='Loan to Value (%):', style=style, layout=layout),
    }

    default_values = {
        'year': 2019, 'loan_limit': 'cf', 'approv_in_adv': 'nopre', 'open_credit': 'nopc',
        'business_or_commercial': 'nob/c', 'neg_ammortization': 'not_neg', 'interest_only': 'not_int',
        'lump_sum_payment': 'not_lpsm', 'construction_type': 'sb', 'occupancy_type': 'pr',
        'secured_by': 'home', 'total_units': '1U', 'credit_type': 'CRIF', 'co-applicant_credit_type': 'EXP',
        'submission_of_application': 'to_inst', 'region': 'south', 'security_type': 'direct',
    }

    predict_button = widgets.Button(description="Predict Loan Status", button_style='success', layout=widgets.Layout(width='90%', margin='20px 0 0 5%'))
    output_area = widgets.Output()

    def on_predict_button_clicked(b):
        with output_area:
            clear_output(wait=True)
            input_data = default_values.copy()
            user_input = {key: widget.value for key, widget in widget_map.items()}
            input_data.update(user_input)
            input_df = pd.DataFrame([input_data])
            print("--- Processing New Application ---")
            input_encoded = pd.get_dummies(input_df)
            input_aligned = input_encoded.reindex(columns=model_columns, fill_value=0)
            prediction = model.predict(input_aligned)
            prediction_proba = model.predict_proba(input_aligned)

            print("\n--- Prediction Result ---")
            if prediction[0] == 0: print("Prediction: Status 0 -> 🟢 Loan Approved")
            else: print("Prediction: Status 1 -> 🔴 Loan Denied")

            print("\n--- Confidence Score ---")
            print(f"Probability of Approval (0): {prediction_proba[0][0]:.2%}")
            print(f"Probability of Denial (1): {prediction_proba[0][1]:.2%}")

    predict_button.on_click(on_predict_button_clicked)

    all_inputs = widgets.VBox(list(widget_map.values()))

    if model_accuracy is not None:
        accuracy_html = f"<div style='background-color: #eef2ff; padding: 15px; border: 2px solid #a5b4fc; border-radius: 10px; margin-bottom: 20px; text-align: center; font-family: Arial Black, sans-serif;'>" \
                        f"<b style='font-size: 1.2em; color: #4f46e5;'>Underlying Model Accuracy:</b> <span style='font-size: 2em; font-weight: bold; color: #3730a3;'>{model_accuracy:.2%}</span></div>"
        display(HTML(accuracy_html))

    print("Interface ready. Please fill in the details and click predict.")
    display(all_inputs, predict_button, output_area)

else:
    print("\nUI could not be built because a model could not be loaded or trained.")



Model files not found. Attempting to train a new model from scratch...

--- Starting On-the-Fly Model Training ---
Training a new Random Forest model...
Model training complete.
Model accuracy calculated: 0.8759
Saving model, columns, and accuracy for this session...
Model assets saved successfully.

Building interactive prediction interface...


Interface ready. Please fill in the details and click predict.


Button(button_style='success', description='Predict Loan Status', layout=Layout(margin='20px 0 0 5%', width='9…

Output()